# Rapport d'évaluation : Comparaison de différentes approches de comptage de mots dans un contexte Big Data

## 1. Introduction et Problématique
Ce rapport présente une étude comparative portant sur l'efficacité de plusieurs méthodes de traitement de données appliquées au problème classique du comptage de mots (*WordCount*). Face aux volumes massifs de données caractéristiques du Big Data (allant de plusieurs gigaoctets à plusieurs téraoctets), le choix de l'architecture et de la méthode de calcul impacte directement les performances et les temps d'exécution.

## 2. Méthodologies évaluées
Trois approches distinctes sont analysées et confrontées dans le cadre de cette étude :
* **Approche séquentielle (Python) :** Une méthode linéaire et simple, idéale pour de petits volumes, mais limitée par l'absence de parallélisme.
* **Pseudo-MapReduce (Python) :** Une émulation logicielle en Python reproduisant séquentiellement les étapes fondamentales du paradigme MapReduce (**Map**, **Shuffle**, et **Reduce**).
* **Hadoop MapReduce :** Une solution distribuée et native exploitant le parallélisme massif pour traiter de grands ensembles de données.

## 3. Protocole expérimental et Volumétrie
Pour mesurer l'évolutivité (*scalabilité*) et les performances de chaque méthode, l'expérimentation s'appuie sur quatre jeux de données textuels dont la taille varie selon le nombre de lignes (n) :
* $n = 100$ lignes
* $n = 100\,000$ lignes
* $n = 5\,000\,000$ lignes
* $n = 50\,000\,000$ lignes

En faisant varier cette volumétrie, l'objectif est d'identifier les limites opérationnelles de chaque approche et de déterminer à partir de quel seuil de données l'utilisation d'un framework distribué devient indispensable.

# I. Version 500 mots

## I.A) Version 1: Python simple

### Génération du fichier de mots

In [65]:
import time
#Création du fichier mots.txt contenant 500 mots

# Liste de 500 mots
mots_base = ["chat", "chien", "oiseau", "voitures", "kiwi"]

# Création du fichier mots.txt avec 500 lignes
with open("mots500.txt", "w", encoding="utf-8") as f:
    for i in range(500):
        # On choisit un mot de la liste de manière cyclique
        mot = mots_base[i % len(mots_base)]
        f.write(mot + "\n")

print("Le fichier mots500.txt a été généré avec succès (500 lignes) !")

Le fichier mots500.txt a été généré avec succès (500 lignes) !


### Comptage des occurences:

In [66]:
#Fonction de comptage des occurences par python simple
def compter_occurrences(fichier):
    with open(fichier, 'r', encoding='utf-8') as f:
        mots = f.read().split()

    compteur = {}
    for mot in mots:
        compteur[mot] = compteur.get(mot, 0) + 1
    return compteur



In [67]:
if __name__ == "__main__":
    fichier = "mots500.txt"
    resultats = compter_occurrences(fichier)
    for mot, nb in sorted(resultats.items(), key=lambda x: x[1], reverse=True):
        print(f"{mot} : {nb}")

chat : 100
chien : 100
oiseau : 100
voitures : 100
kiwi : 100


### Chronométrage du python simple:

In [68]:
def mesurer(fonction, fichier):
    debut = time.perf_counter()
    resultat = fonction(fichier)
    return resultat, time.perf_counter() - debut



In [69]:
mesurer(compter_occurrences, fichier)

({'chat': 100, 'chien': 100, 'oiseau': 100, 'voitures': 100, 'kiwi': 100},
 0.0007286000181920826)

## I.B) Version 2: Pseudo-MapReduce Python et Chronométrage du temps d'exécution

### Création des fonctions 

In [70]:
from collections import defaultdict

#Création de fonction Pseud-mapReduce
def mapper(ligne):
    mots = ligne.strip().split()
    return [(mot, 1) for mot in mots]

def shuffle(mapped_data):
    grouped = defaultdict(list)
    for mot, valeur in mapped_data:
        grouped[mot].append(valeur)
    return grouped

def reducer(grouped_data):
    reduced = {}
    for mot, valeurs in grouped_data.items():
        reduced[mot] = sum(valeurs)
    return reduced



In [57]:
#exécution et chronomètrage
if __name__ == "__main__":
    fichier = "mots500.txt"
    
    debut = time.perf_counter()
    
    mapped = []
    with open(fichier, "r", encoding="utf-8") as f:
        for ligne in f:
            mapped.extend(mapper(ligne))

    grouped = shuffle(mapped)
    reduced = reducer(grouped)

    fin = time.perf_counter()
    
    for mot, count in sorted(reduced.items(), key=lambda x: x[1], reverse=True):
         print(f"{mot} : {count}")
    print(f"\n[Temps d'exécution total] : {fin - debut:.6f} secondes")

chat : 100
chien : 100
oiseau : 100
voitures : 100
kiwi : 100

[Temps d'exécution total] : 0.001153 secondes


# II. Version 2500 mots

In [83]:
# Liste de 2500 mots
mots_base = ["chat", "chien", "oiseau", "voitures", "kiwi"]

# Création du fichier mots.txt avec 5000 lignes
with open("mots2500.txt", "w", encoding="utf-8") as f:
    for i in range(25000000):
        # On choisit un mot de la liste de manière cyclique
        mot = mots_base[i % len(mots_base)]
        f.write(mot + "\n")

print("Le fichier mots2500.txt a été généré avec succès (25000000 lignes) !")

Le fichier mots2500.txt a été généré avec succès (25000000 lignes) !


## II.A) Version 1: Python simple

In [84]:
if __name__ == "__main__":
    fichier = "mots2500.txt"
    resultats = compter_occurrences(fichier)
    for mot, nb in sorted(resultats.items(), key=lambda x: x[1], reverse=True):
        print(f"{mot} : {nb}")

mesurer(compter_occurrences, fichier)

chat : 5000000
chien : 5000000
oiseau : 5000000
voitures : 5000000
kiwi : 5000000


({'chat': 5000000,
  'chien': 5000000,
  'oiseau': 5000000,
  'voitures': 5000000,
  'kiwi': 5000000},
 9.561722499987809)

## II.B) Version 2: Pseudo-MapReduce Python et Chronométrage du temps d'exécution

In [32]:
if __name__ == "__main__":
    fichier = "mots2500.txt"
    
    debut = time.perf_counter()
    
    mapped = []
    with open(fichier, "r", encoding="utf-8") as f:
        for ligne in f:
            mapped.extend(mapper(ligne))

    grouped = shuffle(mapped)
    reduced = reducer(grouped)

    fin = time.perf_counter()
    
    for mot, count in sorted(reduced.items(), key=lambda x: x[1], reverse=True):
         print(f"{mot} : {count}")
    print(f"\n[Temps d'exécution total] : {fin - debut:.6f} secondes")



chat : 200
chien : 200
oiseau : 200
voitures : 200
kiwi : 200

[Temps d'exécution total] : 0.001892 secondes


# III. Version 25000000 mots

In [73]:
# Liste de 5000 mots
mots_base = ["chat", "chien", "oiseau", "voitures", "kiwi"]

# Création du fichier mots.txt avec 5000 lignes
with open("mots25000000.txt", "w", encoding="utf-8") as f:
    for i in range(25000000):
        # On choisit un mot de la liste de manière cyclique
        mot = mots_base[i % len(mots_base)]
        f.write(mot + "\n")

print("Le fichier mots25000000.txt a été généré avec succès (25000000 lignes) !")

Le fichier mots25000000.txt a été généré avec succès (25000000 lignes) !


## III.A) Version 1: Python simple

In [90]:
if __name__ == "__main__":
    fichier = "mots25000000.txt"
    resultats = compter_occurrences(fichier)
    for mot, nb in sorted(resultats.items(), key=lambda x: x[1], reverse=True):
        print(f"{mot} : {nb}")

mesurer(compter_occurrences, fichier)

chat : 5000000
chien : 5000000
oiseau : 5000000
voitures : 5000000
kiwi : 5000000


({'chat': 5000000,
  'chien': 5000000,
  'oiseau': 5000000,
  'voitures': 5000000,
  'kiwi': 5000000},
 10.320084700011648)

## III.B) Version 2: Pseudo-MapReduce Python et Chronométrage du temps d'exécution

In [75]:
#exécution et chronométrage
if __name__ == "__main__":
    fichier = "mots25000000.txt"
    
    debut = time.perf_counter()
    
    mapped = []
    with open(fichier, "r", encoding="utf-8") as f:
        for ligne in f:
            mapped.extend(mapper(ligne))

    grouped = shuffle(mapped)
    reduced = reducer(grouped)

    fin = time.perf_counter()
    
    for mot, count in sorted(reduced.items(), key=lambda x: x[1], reverse=True):
         print(f"{mot} : {count}")
    print(f"\n[Temps d'exécution total] : {fin - debut:.6f} secondes")

chat : 5000000
chien : 5000000
oiseau : 5000000
voitures : 5000000
kiwi : 5000000

[Temps d'exécution total] : 33.202813 secondes


# IV. Version 500000000 mots

In [86]:
with open("mots50000000.txt", "w", encoding="utf-8") as f:
    for i in range(50000000):
        # On choisit un mot de la liste de manière cyclique
        mot = mots_base[i % len(mots_base)]
        f.write(mot + "\n")

print("Le fichier mots50000000.txt a été généré avec succès (50000000 lignes) !")

Le fichier mots50000000.txt a été généré avec succès (50000000 lignes) !


## IV.A) Version 1: Python simple

In [87]:
if __name__ == "__main__":
    fichier = "mots50000000.txt"
    resultats = compter_occurrences(fichier)
    for mot, nb in sorted(resultats.items(), key=lambda x: x[1], reverse=True):
        print(f"{mot} : {nb}")

mesurer(compter_occurrences, fichier)

chat : 10000000
chien : 10000000
oiseau : 10000000
voitures : 10000000
kiwi : 10000000


({'chat': 10000000,
  'chien': 10000000,
  'oiseau': 10000000,
  'voitures': 10000000,
  'kiwi': 10000000},
 19.226681600004667)

## IV.B) Version 2: Pseudo-MapReduce Python et Chronométrage du temps d'exécution

In [89]:
#exécution et chronométrage
if __name__ == "__main__":
    fichier = "mots50000000.txt"
    
    debut = time.perf_counter()
    
    mapped = []
    with open(fichier, "r", encoding="utf-8") as f:
        for ligne in f:
            mapped.extend(mapper(ligne))

    grouped = shuffle(mapped)
    reduced = reducer(grouped)

    fin = time.perf_counter()
    
    for mot, count in sorted(reduced.items(), key=lambda x: x[1], reverse=True):
         print(f"{mot} : {count}")
    print(f"\n[Temps d'exécution total] : {fin - debut:.6f} secondes")

chat : 10000000
chien : 10000000
oiseau : 10000000
voitures : 10000000
kiwi : 10000000

[Temps d'exécution total] : 82.758807 secondes


# V) Lancement de Hadoop MapReduce et pyspark


In [ ]:


from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, lower, regexp_replace, split

# 1. Créer une session Spark
spark = SparkSession.builder \
    .appName("WordCountMotsTxt") \
    .master("local[*]") \
    .getOrCreate()

# 2. Charger le fichier texte dans un DataFrame Spark
# Chaque ligne du fichier devient une ligne dans le DataFrame avec la colonne "value"
# mots500 à modifier

df = spark.read.text("mots500.txt")

# 3. Traiter le texte :
# - Mettre tout en minuscules (optionnel)
# - Retirer la ponctuation pour éviter les doublons (ex: "mot," vs "mot")
# - Diviser chaque ligne en un tableau de mots en se basant sur les espaces
# - Utiliser explode pour transformer chaque élément du tableau en une ligne distincte
words_df = df.select(
    explode(
        split(
            regexp_replace(lower(col("value")), "[^a-zA-Zàâäéèêëîïôöùûüç\\s]", ""), 
            "\\s+"
        )
    ).alias("word")
).filter(col("word") != "") # Filtrer les lignes vides

# 4. Compter les occurrences de chaque mot
word_counts = words_df.groupBy("word") \
    .count() \
    .orderBy(col("count").desc())

# 5. Afficher les résultats (par exemple, les 20 mots les plus fréquents)
word_counts.show(20, truncate=False)

# (Optionnel) Sauvegarder les résultats dans un fichier CSV ou texte
# word_counts.write.csv("resultat_comptage.csv", header=True)

# 6. Arrêter la session Spark
spark.stop()

# Tableau récapitulatif des scores

| Nombre de mots | Python | Pseudo-MapReduce | MapReduce | PySpark |
| :--- | :---: | :---: | :---: | ---: |
| 500 | 0.00072 | 0.0011 | 6.568 | 0.178 |
| 2500 | 9.561 | 0.0018 | 12.524 | 0.258 |
| 25000000 | 9.534 | 33.202 | 51.578 | 0.784 |
| 50000000 | 19.226 | 82.758 | 86.445 | 7.256 |

## 4. Conclusion et Analyse des Résultats

L'analyse des performances issues de notre expérimentation met en évidence des comportements très contrastés selon la volumétrie des données et l'approche choisie :

* **Pour les petits volumes (500 à 2 500 mots) :** 
  Les approches logicielles en Python (séquentielle et Pseudo-MapReduce) dominent nettement avec des temps d'exécution quasi instantanés (inférieurs à la seconde ou de quelques secondes). À l'inverse, les frameworks lourds comme Hadoop MapReduce subissent une forte surcharge initiale (*overhead* lié au démarrage de la machine virtuelle Java et du framework), ce qui rend leur utilisation contre-productive sur de petits ensembles de données.

* **Pour les volumes massifs (25 000 000 à 50 000 000 de mots) :**
  Les limites des approches séquentielles et de l'émulation apparaissent clairement : le Pseudo-MapReduce voit ses temps de calcul exploser (jusqu'à 82,7 secondes pour 50 millions de mots) en raison de sa nature séquentielle non optimisée. Hadoop MapReduce voit également ses performances se dégrader face à la charge.

* **La supériorité de PySpark :**
  PySpark se positionne comme la solution la plus performante et la plus stable sur les grands volumes (seulement 7,2 secondes pour 50 millions de mots). Cela illustre l'efficacité des moteurs de traitement distribués et optimisés en mémoire par rapport au MapReduce traditionnel.

**Bilan global :** 
Il n'existe pas de solution universelle. Pour de faibles volumes, un script Python classique reste le choix le plus simple et rapide. En revanche, dès que le volume de données franchit un seuil critique, le recours à des technologies de traitement distribué (comme PySpark) devient indispensable pour garantir l'évolutivité et des temps de traitement acceptables.